In [ ]:
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer
import numpy as np
from pathlib import Path
import pickle

In [ ]:
raw_csv_path = "../data/raw/AI_Human.csv"
df = pd.read_csv(raw_csv_path)

In [ ]:
total_len = len(df)
total_ai = (df["generated"] == 1).sum()
total_human = (df["generated"] == 0).sum()

print(f"The dataset consists of {total_len} samples, of which {total_ai} are ai generated and {total_human} are human written.")

In [ ]:
df["text_len"] = df["text"].apply(lambda x: len(x))
df["word_count"] = df["text"].str.count(" ") + 1
median_char = df["text_len"].median().item()
median_word = df["word_count"].median().item()
print(f"Median word count equals {median_word} while median character count {median_char}")

In [ ]:
fig = px.histogram(
    df["text_len"],
    labels={
        "value": "Length of a specific entry",
        "count": "Count",
    },
)

fig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_function(examples):
    return tokenizer(
        examples,
        padding="max_length",
        truncation=True,
        max_length=512
    )

df_subset = df.sample(10_000)
tokenized = tokenize_function(list(df_subset["text"].to_numpy()))
tokenized = np.array(tokenized["attention_mask"])
df_subset["token_size"] = tokenized.sum(axis=1)

fig = px.scatter(df_subset, "text_len", "token_size", labels={"token_size": "Number of generated tokens", "text_len":"Length of the text"})
fig

In [ ]:
df_subset["pred_token_size"] = df_subset["text_len"] / 5
tokens_lost = (df_subset["pred_token_size"] - df_subset["token_size"]).sum()# * len(df)/10_000
tokens_kept = df_subset["token_size"].sum()
tokens_total_pred = df_subset["pred_token_size"].sum()
truncate_loss = tokens_kept / (tokens_kept + tokens_lost)

print(f"We loose about {truncate_loss}")


In [ ]:
model_dir = "models"


model_files = [file for file in Path.iterdir(Path(f"../{model_dir}")) if not Path.is_dir(file)]

model_dict = {}
for file in model_files:
    with open(file, "rb") as f:
        model_dict[file] = pickle.load(f)
model_dict

def get_feature_importances(model):
    if hasattr(model, "coef_"):
        return model.coef_.ravel()
    elif hasattr(model, "feature_importances_"):
        return model.feature_importances_


df = (
    pd.DataFrame(
        {
            model_path.stem: get_feature_importances(model)
            for model_path, model in model_dict.items()
        }
    )
    .reset_index(drop=False)
    .rename(columns={"index": "feature_num"})
)
df.head()

In [ ]:
df_corr = df.drop(columns="feature_num").corr(method="spearman")
fig = px.imshow(
    df_corr,
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
)
fig.data[0].text = df_corr.round(3).to_numpy()
fig.data[0].texttemplate = "%{text}"
fig.update_traces(textfont_size=18)
fig